# Spherical Fidelity Plots

This notebook generates the two polar plots from the manuscript showing which metric
operators pass or fail the adversarial threshold, visualised as points on a sphere
parameterised by $(r, \\theta, \\phi)$:

- **Vertical plane**: meridional slice with $\\phi = -\\pi/2$ and $\\phi = \\pi/2$,
  showing how the threshold outcome varies with $\\theta$.
- **Horizontal plane**: equatorial slice at $\\theta = \\pi/2$, showing how the
  threshold outcome varies with $\\phi$.

Pass/fail results are read from `data/processed/experimental_fidelities_master.csv`.
Figures are saved to `figures/`.

In [ ]:
import os
from math import pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameters

In [ ]:
# Select which experiment to plot: 'qutrit' or 'qubit'
which_experiment = 'qutrit'

INPUT_PATH = "data/processed/experimental_fidelities_master.csv"
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Metric operator parameter grid

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Load pass/fail results and prepare plot data

In [ ]:
ETA_GRID = eta_grid()
ETA_GRID_arr = np.array(ETA_GRID)

radius = np.unique(ETA_GRID_arr[:, 0])
theta  = np.unique(ETA_GRID_arr[:, 1])

# Load pass/fail column for the selected experiment
df = pd.read_csv(INPUT_PATH)
passed_col = 'qutrit_passed' if which_experiment == 'qutrit' else 'qubit_passed'
verify_res = df[passed_col].tolist()

# Separate points into pass/fail/boundary categories
# Points at radius[2] (r = 0.333) pass only for qutrit, not qubit
true_x,  true_y,  true_z  = [], [], []
false_x, false_y, false_z = [], [], []
onlyqutrit_x, onlyqutrit_y, onlyqutrit_z = [], [], []

for j, point in enumerate(ETA_GRID):
    x = point[0] * np.sin(point[1]) * np.cos(point[2])
    y = point[0] * np.sin(point[1]) * np.sin(point[2])
    z = point[0] * np.cos(point[1])
    if point[0] == radius[2]:  # r = 0.333: passes qutrit threshold only
        onlyqutrit_x.append(x); onlyqutrit_y.append(y); onlyqutrit_z.append(z)
    elif verify_res[j]:
        true_x.append(x);  true_y.append(y);  true_z.append(z)
    else:
        false_x.append(x); false_y.append(y); false_z.append(z)

# Horizontal plane: theta = pi/2 fixed, phi varies
horizontal_plane = np.array([
    [ETA_GRID[i][0], ETA_GRID[i][2]]
    for i in range(len(ETA_GRID)) if ETA_GRID[i][1] == theta[1]
])

# Vertical plane: phi = -pi/2 and pi/2 fixed, theta varies
vertical_plane = np.array([
    [ETA_GRID[i][0], ETA_GRID[i][1]]
    for i in range(len(ETA_GRID)) if ETA_GRID[i][1] in (theta[0], theta[2])
])

# Split each plane into pass/fail/boundary subsets
def split_plane(plane):
    passed  = plane[plane[:, 0] < radius[2]]
    failed  = plane[plane[:, 0] > radius[2]]
    boundary = plane[plane[:, 0] == radius[2]]
    return passed, failed, boundary

true_h, false_h, qutrit_h = split_plane(horizontal_plane)
true_v, false_v, qutrit_v = split_plane(vertical_plane)

## Vertical plane plot

Meridional slice with $\\phi = -\\pi/2$ (left) and $\\phi = \\pi/2$ (right), showing
threshold outcomes as $\\theta$ varies.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))

ax.scatter(true_v[:, 1],   true_v[:, 0],   c='c', label='Passed the threshold')
ax.scatter(false_v[:, 1],  false_v[:, 0],  c='m', label='Failed the threshold')
ax.scatter(qutrit_v[:, 1], qutrit_v[:, 0],
           c='c' if which_experiment == 'qutrit' else 'm')

ax.set_theta_offset(np.pi / 2)
ax.set_xticks(np.arange(0, 2*np.pi, np.pi/4))
ax.set_xticklabels([
    r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
    r'$\pi$', r'$3\pi/4$', r'$\pi/2$', r'$\pi/4$'
])

panel = '(b)' if which_experiment == 'qubit' else '(a)'
exp   = 'Qubit' if which_experiment == 'qubit' else 'Qutrit'
ax.set_title(
    fr"{panel}  {exp} Vertical Plane (varying $\theta$)" + "\n"
    + r"Left: $\phi=-\pi/2$  |  Right: $\phi=\pi/2$",
    loc='center', fontweight='bold', pad=15
)
ax.plot([0.5, 0.5], [0, 1], color='black', linestyle='--',
        transform=ax.transAxes, clip_on=False)
ax.set_rmax(1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(80)
ax.grid(True)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False, fontsize=11)

plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, f"{which_experiment}_vertical")
plt.savefig(fname + '.png', bbox_inches='tight')
plt.savefig(fname + '.svg', bbox_inches='tight')
plt.show()
print(f"Saved to {fname}.png / .svg")

## Horizontal plane plot

Equatorial slice at $\\theta = \\pi/2$, showing threshold outcomes as $\\phi$ varies.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))

ax.scatter(true_h[:, 1],   true_h[:, 0],   c='c', label='Passed the threshold')
ax.scatter(false_h[:, 1],  false_h[:, 0],  c='m', label='Failed the threshold')
ax.scatter(qutrit_h[:, 1], qutrit_h[:, 0],
           c='c' if which_experiment == 'qutrit' else 'm')

ax.set_theta_offset(np.pi / 2)
ax.set_xticks(np.arange(0, 2*np.pi, np.pi/4))
ax.set_xticklabels([
    r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
    r'$\pi$', r'$5\pi/4$', r'$3\pi/2$', r'$7\pi/4$'
])

panel = '(b)' if which_experiment == 'qubit' else '(a)'
exp   = 'Qubit' if which_experiment == 'qubit' else 'Qutrit'
ax.set_title(
    fr"{panel}  {exp} Horizontal Plane (varying $\phi$)" + "\n"
    + r"Equatorial Plane: $\theta=\pi/2$",
    loc='center', fontweight='bold', pad=15
)
ax.set_rmax(1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(80)
ax.grid(True)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False, fontsize=11)

plt.tight_layout()
fname = os.path.join(OUTPUT_DIR, f"{which_experiment}_horizontal")
plt.savefig(fname + '.png', bbox_inches='tight')
plt.savefig(fname + '.svg', bbox_inches='tight')
plt.show()
print(f"Saved to {fname}.png / .svg")